# Build the Training Swath Projection Matrix

Curated from the archived research notebook `2. Save H_swath(training).ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

Full preprocessing requires input files and an `internal_waves` module that
were not included in the available collection. See `docs/reproduction.md`.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='preprocessing')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import xarray as xr
import numpy as np
import scipy
import cmocean as cmo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from glob2 import glob
import dask.array
from swath_rossby_wave import inversion
from tqdm import tqdm
from swath_rossby_wave import skill_matrix, build_h_matrix2, build_hswath_matrix2, inversion, make_error_over_time
import matplotlib.patches as mpatches
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from numpy import linalg as LA


In [ ]:
day0, day1 = 0, 200
n_waves = '190' #number of waves
day0_array = np.arange(0, 84 * 117, 84)

MModes = 1 # Rossby wave vertical modes
wave_files = input_glob('./new_training_data_rossby_wave_estimate_*_' + n_waves +'waves_swotdomain_'+ str(int((day1 - day0))) +'days.nc')
wave_files = sorted(wave_files)

lonidx_west, lonidx_east  =  76, 112
latidx_south, latidx_north = 27, 67

new_data = xr.open_dataset(input_path('./aviso_tot_MSLA_ccs_data.nc'))
dsave = new_data.dsave.values
tsave = new_data.tsave.values
xsave = new_data.xsave.values
ysave = new_data.ysave.values
# read in AVISO/Copernicus SSHA data and use to set mask
dsave_transpose = np.transpose(dsave, (1, 0, 2))
SSHA = dsave_transpose[latidx_south:latidx_north, lonidx_west:lonidx_east, :]
T_time = tsave * 86400 # in seconds
T_time = T_time.flatten()
lon, lat = (360 - xsave[lonidx_west:lonidx_east]) * -1, ysave[latidx_south:latidx_north]
dlon = lon - lon.mean()
dlat = lat - lat.mean()
tsave_flat = tsave.flatten()
date_time_all = np.array([np.datetime64(int(atime - tsave_flat[0] + 8401), 'D') for atime in tsave_flat])
ssha_time_mean = SSHA[:, :, : ].mean(axis = -1) # remove multi-year mean (climatology)
ssha_time_mean_expanded = ssha_time_mean[:, :, np.newaxis]
# remove mean from SSH data to produce anomaly over full analysis period
SSHA = SSHA - ssha_time_mean_expanded
#  alternately could remove 80-day mean  SSHA[day0 + day0 + 30].mean(axis = -1)
#  this is not recommended
SSHA_masked = np.ma.masked_invalid(SSHA)
ssha_mask = np.ma.getmask(SSHA_masked)

# set parameters for Rossby wave propagationn model
Phi0 = lat.mean() # central latitude (φ0)
Omega = 7.27e-5 # Ω is the angular speed of the earth
Earth_radius = 6.371e6 / 1e5 # meters
Beta = 2 * Omega * np.cos(Phi0*np.pi/180.) / Earth_radius
f0 = 2 * Omega * np.sin(Phi0*np.pi/180.) #1.0313e-4 # 45 N


In [ ]:
# identify SWOT tracks---here using only 2 sample tracks from one-day repeat orbit
swot_files = sorted(input_glob('./SWOT_L2_LR_SSH_Expert_474*.nc'))
swot_ds = xr.open_mfdataset(swot_files, combine='nested', concat_dim = 'num_lines') # , engine='store', chunks={'time': 10})
# Extract the nadir position for the ground track
latitude_nadir = swot_ds['latitude_nadir'].values
longitude_nadir = swot_ds['longitude_nadir'].values-360

# Extract the latitudes and longitudes within the swath
latitude = swot_ds['latitude'].values
longitude = swot_ds['longitude'].values-360

# read cross track distance, and depth
cross_track = swot_ds['cross_track_distance'].values
depth=swot_ds['depth_or_elevation'].values

# use a brute-strength approach to separate ascending and descending tracks
direction = np.sign(np.diff(latitude,n=1,axis=0))

# to tightly distinguish ascending and descening, use time difference between reference tracks
time_swot = swot_ds['time'].values
time_hrs=time_swot[:].astype('datetime64[h]').astype(int)-466680
direction_asc = np.all([time_hrs >5,time_hrs < 11],axis=0)
direction_des = np.all([time_hrs >=11,time_hrs < 22],axis=0)


In [ ]:
# identify points to use within swath and remove swath points that won't be used
mask=np.zeros([latitude.shape[0],latitude.shape[1]])
deltax=np.zeros([latitude.shape[0],latitude.shape[1]])
asc_des=np.zeros([latitude.shape[0],latitude.shape[1]])

# chose cross-track positions
#  original code used range(2,latitude.shape[1],8), but here the indices are hard-wired in order to skip the gap at nadir
# use=range(2,len(counter),8)
use2=[2, 10, 18, 26,42,50,58,66]

deltax=cross_track/1.e5 # distance in 100 km, relative to nadir
asc_des=(direction+1)/2.  ## 1 = ascending; 0 = descending
for i in range(0,latitude.shape[0]-1,16):
    for j in use2:
        if(latitude[i,j]>=min(lat) and latitude[i,j]<=max(lat)
           and longitude[i,j]<=max(lon) and longitude[i,j]>=min(lon)):
            mask[i,j]=1

index=np.where(mask==1)


In [ ]:
# load stratification, taken from numerical model
strat_ds = xr.open_dataset(input_path('./stratification_sample_ccs.nc'))
# parameters for Rossby wave model
Psi = strat_ds.Psi.data
Rm = 5e4  / 1e5 # 50 km to degree
wavespeed = Rm * f0  # deg / s strat_ds.C2[:MModes].data
Rm = np.array([Rm]) #unit: degree


In [ ]:
# set zonal and meridional wavenumber increments and upper/lower bounds
# nominally assume a 10 x 10 degree domain, though we actually use a slightly rectangular domain
L_lat = 10 # domain latitude length degree
L_lon = 10 # domain lognitude length

domain_factor = 1.1 # the smaller, the less waves

l_interval = 2 * np.pi / (domain_factor * L_lat) # zonal wavemenumber interval
k_interval = 2 * np.pi / (domain_factor * L_lon) # longitutional wavemenumber interval

lambda_min = 1.2 # 100km = 1 degree minimum wavelength resolved , the smaller, the more waves

k_min = 0
k_max = 2 * np.pi / lambda_min
l_max = k_max
l_min = -1 * k_max

# set range of k and l
k_n_orig = np.arange(k_min, k_max, k_interval) # degree^-1
l_n_orig = np.arange(l_min, l_max, l_interval) # degree^-1
l_n = l_n_orig.reshape(len(l_n_orig), MModes) #* 0 # lon, zonal propagration
k_n = k_n_orig.reshape(len(k_n_orig), MModes) #* 0 # lat, meridonal propagration
# set size of wavenumber domain
M = k_n.size * l_n.size


In [ ]:
# create a dummy masked matrix in order to build the full H matrix
day0 = 0
day1 = 200
MSLA0 = SSHA_masked[:, :, day0:day1] #AVISO input
ssha_clean = np.ma.masked_invalid(np.zeros([MSLA0.shape[0],MSLA0.shape[1],MSLA0.shape[2]]))
# define an H matrix for swath points only
H_swath = build_hswath_matrix2(ssha_clean, MModes, k_n, l_n, lon,lat,longitude, latitude, index, T_time, Psi, Rm, day0-day0)


In [ ]:
ds = xr.Dataset(
    {
        "H_swath": (["dimension*day", "Wavenumber*2"], H_swath)
    }
)

# Save the Dataset to a NetCDF file
ds.to_netcdf(output_path("new_H_swath.nc"))
